# Exercise 4.1: Routing Features from OpenRouteService in Mainz

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yfeng-hsm/KI_Geodatenanalyse_SS26/blob/main/lectures/04_spatial_data_analysis/notebooks/exercise_4_1_openrouteservice_routing_mainz.ipynb)

This feature-acquisition exercise uses OpenRouteService to calculate route distance, duration, and geometry features in Mainz. These routing features can enrich downstream geospatial learning tasks.

## Learning Outcomes

- Call the OpenRouteService directions API to acquire route features.
- Compare walking and cycling routes between Mainz locations.
- Inspect route distance and duration as model-ready contextual features.
- Visualize route geometries as spatial feature context on a Folium map.
- Generate an accessibility feature table from one origin to nearby POI classes.
- Run simple plausibility tests for route results.

## 1. Colab Setup

In [ ]:
!pip -q install requests folium shapely geopandas pandas


## 2. Imports and Known Mainz Places

In [ ]:
from __future__ import annotations

import getpass
import json
import math
from typing import Any

import folium
import pandas as pd
import requests
from IPython.display import Markdown, display
from shapely.geometry import shape

MAINZ_CENTER = (49.9929, 8.2473)
KNOWN_PLACES = {
    "mainz_hbf": {"label": "Mainz Hauptbahnhof", "lat": 50.0010, "lon": 8.2587},
    "jgu": {"label": "Johannes Gutenberg University Mainz", "lat": 49.9936, "lon": 8.2419},
    "mainz_dom": {"label": "Mainz Cathedral", "lat": 49.9995, "lon": 8.2742},
    "mainz_theater": {"label": "Staatstheater Mainz", "lat": 50.0002, "lon": 8.2714},
}

for place_id, place in KNOWN_PLACES.items():
    print(place_id, "->", place["label"], (place["lat"], place["lon"]))


## 3. OpenRouteService API Key

Create a free developer key at https://openrouteservice.org/dev/#/signup and paste it below. Do not save API keys in the notebook.

In [ ]:
ORS_API_KEY = getpass.getpass("Paste your OpenRouteService API key for this session: ").strip()
if not ORS_API_KEY:
    raise RuntimeError("No ORS API key entered.")
print("ORS API key loaded for this notebook session.")


## 4. Route Request Helper

In [ ]:
ORS_BASE_URL = "https://api.openrouteservice.org/v2/directions"


def run_ors_route(start_place_id: str, end_place_id: str, profile: str = "foot-walking") -> dict[str, Any]:
    if profile not in {"foot-walking", "cycling-regular", "driving-car"}:
        raise ValueError("Unsupported profile.")
    start = KNOWN_PLACES[start_place_id]
    end = KNOWN_PLACES[end_place_id]
    url = f"{ORS_BASE_URL}/{profile}/geojson"
    payload = {"coordinates": [[start["lon"], start["lat"]], [end["lon"], end["lat"]]]}
    response = requests.post(
        url,
        headers={"Authorization": ORS_API_KEY, "Content-Type": "application/json"},
        json=payload,
        timeout=60,
    )
    if not response.ok:
        print(response.text[:1500])
        response.raise_for_status()
    return response.json()


def route_summary(route_json: dict[str, Any]) -> dict[str, Any]:
    feature = route_json["features"][0]
    summary = feature["properties"].get("summary", {})
    return {
        "distance_m": float(summary.get("distance", 0)),
        "duration_s": float(summary.get("duration", 0)),
        "geometry": shape(feature["geometry"]),
    }


## 5. Example Route: Mainz Hbf to JGU

In [ ]:
route_json = run_ors_route("mainz_hbf", "jgu", profile="foot-walking")
summary = route_summary(route_json)
print(f"Distance: {summary['distance_m'] / 1000:.2f} km")
print(f"Duration: {summary['duration_s'] / 60:.1f} minutes")
print("Raw route feature keys:", route_json["features"][0].keys())


## 6. Map the Route

In [ ]:
def display_route_map(route_json: dict[str, Any], start_place_id: str, end_place_id: str, title: str) -> folium.Map:
    summary = route_summary(route_json)
    m = folium.Map(location=MAINZ_CENTER, zoom_start=13, tiles="OpenStreetMap", control_scale=True)
    html = f'<div style="position: fixed; top: 10px; left: 50px; z-index: 9999; background: white; padding: 8px 10px; border: 1px solid #999; font-size: 14px;"><strong>{title}</strong></div>'
    m.get_root().html.add_child(folium.Element(html))

    start = KNOWN_PLACES[start_place_id]
    end = KNOWN_PLACES[end_place_id]
    folium.Marker((start["lat"], start["lon"]), popup=start["label"], icon=folium.Icon(color="green")).add_to(m)
    folium.Marker((end["lat"], end["lon"]), popup=end["label"], icon=folium.Icon(color="red")).add_to(m)

    coords = [(lat, lon) for lon, lat in summary["geometry"].coords]
    folium.PolyLine(coords, color="blue", weight=5, opacity=0.8).add_to(m)
    display(Markdown(f"Distance: {summary['distance_m'] / 1000:.2f} km. Duration: {summary['duration_s'] / 60:.1f} minutes."))
    return m


display_route_map(route_json, "mainz_hbf", "jgu", "Walking route: Mainz Hbf to JGU")


## 7. Plausibility Tests

In [ ]:
assert 1500 <= summary["distance_m"] <= 7000, "Distance should be plausible for Mainz Hbf to JGU."
assert summary["duration_s"] > 0, "Duration should be positive."
assert not summary["geometry"].is_empty, "Route geometry should not be empty."
print("Route plausibility checks passed.")


## 8. Feature Block: JGU Accessibility to Nearby POI Classes

This block turns routing into model-ready features. Starting from JGU, it selects the nearest candidate POI in three teaching categories and calculates route distance and duration for walking, cycling, and driving.

The public-transport feature is handled as an access proxy: the hosted OpenRouteService API supports walking, cycling, and driving profiles, but not a stable public-transport routing profile. For this exercise, public transport access is represented by the walking route from JGU to the nearest transit stop candidate.

In [ ]:
JGU_ORIGIN = {
    "poi_id": "jgu",
    "label": "Johannes Gutenberg University Mainz",
    "lat": KNOWN_PLACES["jgu"]["lat"],
    "lon": KNOWN_PLACES["jgu"]["lon"],
}

POI_CANDIDATES = [
    # Public transport access points.
    {"poi_id": "mainz_universitaet_stop", "poi_category": "public_transport_stop", "label": "Mainz Universitaet stop", "lat": 49.9938, "lon": 8.2359},
    {"poi_id": "mainz_hbf", "poi_category": "public_transport_stop", "label": "Mainz Hauptbahnhof", "lat": 50.0010, "lon": 8.2587},
    {"poi_id": "mainz_roemisches_theater", "poi_category": "public_transport_stop", "label": "Mainz Roemisches Theater", "lat": 49.9947, "lon": 8.2777},
    # City-center and cultural destinations.
    {"poi_id": "mainz_dom", "poi_category": "city_center_culture", "label": "Mainz Cathedral", "lat": 49.9995, "lon": 8.2742},
    {"poi_id": "mainz_theater", "poi_category": "city_center_culture", "label": "Staatstheater Mainz", "lat": 50.0002, "lon": 8.2714},
    {"poi_id": "gutenberg_museum", "poi_category": "city_center_culture", "label": "Gutenberg Museum", "lat": 49.9990, "lon": 8.2749},
    # Health-care destinations.
    {"poi_id": "unimedizin_mainz", "poi_category": "healthcare", "label": "Universitaetsmedizin Mainz", "lat": 49.9915, "lon": 8.2564},
    {"poi_id": "marienhaus_mainz", "poi_category": "healthcare", "label": "Marienhaus Klinikum Mainz", "lat": 50.0063, "lon": 8.2638},
    {"poi_id": "drk_schmerz_zentrum", "poi_category": "healthcare", "label": "DRK Schmerz-Zentrum Mainz", "lat": 49.9970, "lon": 8.2521},
]

ROUTING_PROFILES = {
    "foot-walking": {"mode": "walking", "color": "#1b9e77"},
    "cycling-regular": {"mode": "cycling", "color": "#377eb8"},
    "driving-car": {"mode": "driving", "color": "#e41a1c"},
}


def haversine_m(lat1: float, lon1: float, lat2: float, lon2: float) -> float:
    radius_m = 6_371_000.0
    phi1 = math.radians(lat1)
    phi2 = math.radians(lat2)
    delta_phi = math.radians(lat2 - lat1)
    delta_lambda = math.radians(lon2 - lon1)
    a = math.sin(delta_phi / 2) ** 2 + math.cos(phi1) * math.cos(phi2) * math.sin(delta_lambda / 2) ** 2
    return radius_m * 2 * math.atan2(math.sqrt(a), math.sqrt(1 - a))


def run_ors_route_between_points(start: dict[str, Any], end: dict[str, Any], profile: str) -> dict[str, Any]:
    url = f"{ORS_BASE_URL}/{profile}/geojson"
    payload = {"coordinates": [[start["lon"], start["lat"]], [end["lon"], end["lat"]]]}
    response = requests.post(
        url,
        headers={"Authorization": ORS_API_KEY, "Content-Type": "application/json"},
        json=payload,
        timeout=60,
    )
    if not response.ok:
        print(response.text[:1500])
        response.raise_for_status()
    return response.json()


poi_candidates = pd.DataFrame(POI_CANDIDATES)
poi_candidates["straight_line_m"] = poi_candidates.apply(
    lambda row: haversine_m(JGU_ORIGIN["lat"], JGU_ORIGIN["lon"], row["lat"], row["lon"]),
    axis=1,
)
nearest_pois = (
    poi_candidates.sort_values("straight_line_m")
    .groupby("poi_category", as_index=False)
    .head(1)
    .sort_values("poi_category")
    .reset_index(drop=True)
)

feature_rows = []
feature_routes = {}
for poi in nearest_pois.to_dict("records"):
    target = {"label": poi["label"], "lat": poi["lat"], "lon": poi["lon"]}
    for profile, profile_meta in ROUTING_PROFILES.items():
        route = run_ors_route_between_points(JGU_ORIGIN, target, profile)
        summary = route_summary(route)
        feature_routes[(poi["poi_category"], profile_meta["mode"])] = route
        feature_rows.append(
            {
                "origin_id": JGU_ORIGIN["poi_id"],
                "origin_label": JGU_ORIGIN["label"],
                "poi_category": poi["poi_category"],
                "nearest_poi_id": poi["poi_id"],
                "nearest_poi_label": poi["label"],
                "mode": profile_meta["mode"],
                "straight_line_m": round(poi["straight_line_m"], 1),
                "route_distance_m": round(summary["distance_m"], 1),
                "route_duration_min": round(summary["duration_s"] / 60, 1),
            }
        )

accessibility_features = pd.DataFrame(feature_rows)

nearest_transit = nearest_pois[nearest_pois["poi_category"] == "public_transport_stop"].iloc[0]
transit_access = accessibility_features[
    (accessibility_features["poi_category"] == "public_transport_stop")
    & (accessibility_features["mode"] == "walking")
].copy()
transit_access["feature_name"] = "public_transport_access_proxy"
transit_access = transit_access[
    [
        "feature_name",
        "origin_id",
        "nearest_poi_id",
        "nearest_poi_label",
        "route_distance_m",
        "route_duration_min",
    ]
]

print("Nearest candidate POI per category:")
display(nearest_pois[["poi_category", "poi_id", "label", "straight_line_m"]])
print("Accessibility feature table:")
display(accessibility_features)
print("Public transport access proxy:")
display(transit_access)

## 9. Map the Accessibility Features

The map shows JGU, the selected nearest POI per category, and the walking/cycling/driving route geometries used to create the feature table. The public-transport proxy is the walking access route to the nearest transit stop.

In [ ]:
feature_map = folium.Map(location=[JGU_ORIGIN["lat"], JGU_ORIGIN["lon"]], zoom_start=13, tiles="OpenStreetMap", control_scale=True)
folium.Marker(
    (JGU_ORIGIN["lat"], JGU_ORIGIN["lon"]),
    popup=JGU_ORIGIN["label"],
    tooltip="Origin: JGU",
    icon=folium.Icon(color="green", icon="play"),
).add_to(feature_map)

category_icon_colors = {
    "public_transport_stop": "blue",
    "city_center_culture": "purple",
    "healthcare": "red",
}
for _, poi in nearest_pois.iterrows():
    folium.Marker(
        (poi["lat"], poi["lon"]),
        popup=f"{poi['poi_category']}<br>{poi['label']}",
        tooltip=f"Nearest {poi['poi_category']}: {poi['label']}",
        icon=folium.Icon(color=category_icon_colors.get(poi["poi_category"], "gray")),
    ).add_to(feature_map)

for profile, profile_meta in ROUTING_PROFILES.items():
    group = folium.FeatureGroup(name=profile_meta["mode"], show=True)
    for poi_category in nearest_pois["poi_category"]:
        route = feature_routes[(poi_category, profile_meta["mode"])]
        summary = route_summary(route)
        coords = [(lat, lon) for lon, lat in summary["geometry"].coords]
        matching_row = accessibility_features[
            (accessibility_features["poi_category"] == poi_category)
            & (accessibility_features["mode"] == profile_meta["mode"])
        ].iloc[0]
        folium.PolyLine(
            coords,
            color=profile_meta["color"],
            weight=4 if profile_meta["mode"] == "walking" else 3,
            opacity=0.75,
            tooltip=(
                f"{profile_meta['mode']} to {matching_row['nearest_poi_label']}: "
                f"{matching_row['route_distance_m'] / 1000:.2f} km, "
                f"{matching_row['route_duration_min']:.1f} min"
            ),
        ).add_to(group)
    group.add_to(feature_map)

folium.LayerControl().add_to(feature_map)
feature_map

## 10. Student Tasks

1. Add one more POI category, for example parks, supermarkets, or administrative offices.
2. Replace the hard-coded POI candidates with POIs queried from OpenStreetMap or another source.
3. Change the origin from JGU to another Mainz location and regenerate the feature table.
4. Compare network distance with straight-line distance and identify where the difference is largest.
5. Explain why the public-transport value here is only an access proxy, and what data would be needed for true transit travel time.